# Chapter 15 — Kernel Fusion

> Course: **llm.c — Zero to Hero**, Chapter 15 of ~20.
> Builds on Chapters 9-13.

In Chapter 14 we saw cuBLAS fuse matmul + bias + GELU into one kernel via the *epilogue*. The same idea applies broadly: **whenever two kernels read/write the same tensor in sequence, fusing them into one kernel saves a round-trip through global memory** — and global memory is usually the bottleneck.

This chapter shows the principle in `llm.c`'s production code: `fused_residual_forward` (residual + LayerNorm in one pass) and `fused_classifier` (softmax + CE forward + backward all together). We won't rewrite either from scratch — they're hundreds of lines — but you'll read the production versions and time fused vs unfused on a small kernel of your own.

### Learning objectives

By the end of this chapter you will:

- Articulate the fusion rule: "if op B's input is op A's output, and you don't need A's output for anything else, fuse them."
- Read `fused_residual_forward.cu` and `fused_classifier.cuh` and explain what they fuse.
- Write a fused vs unfused (`x + y → relu`) demo and measure the speedup yourself.


## 1. Concept — Why Fusion Wins

Consider a Transformer block fragment:

```
y = x + attn_out          # residual_forward
z = layernorm(y)          # layernorm_forward
```

Naive implementation = two kernel launches:

```
Kernel A:   reads x, attn_out (each `B*T*C` floats)
            writes y       (`B*T*C` floats)
            → 3 * B*T*C floats of memory traffic
Kernel B:   reads y       (`B*T*C` floats)
            writes z       (`B*T*C` floats)
            → 2 * B*T*C floats of memory traffic
            Plus reductions over each row.
TOTAL:      5 * B*T*C floats of bandwidth, 2 launches
```

Fused: kernel A keeps `y` in registers / shared memory and immediately runs LayerNorm on it.

```
Fused kernel: reads x, attn_out
              computes y in registers
              computes mean, var, rstd directly from y in registers
              writes z
              → 3 * B*T*C floats of memory traffic, no `y` in DRAM
TOTAL:        3 * B*T*C floats (40% reduction)
```

For bandwidth-bound ops (which most non-matmul Transformer ops are), this *directly* translates to ~40% less time. And we save a kernel launch (~10 μs of overhead).

The catch: the fused kernel is **harder to read** and **harder to reuse**. Fusion is a tradeoff between speed and modularity, and the LLM training community has decided that the speedup is worth it.


## 2. `fused_residual_forward` in `llm.c`

[`dev/cuda/fused_residual_forward.cu`](dev/cuda/fused_residual_forward.cu) and the production [`llmc/layernorm.cuh`](llmc/layernorm.cuh) both contain `fused_residual_forward_kernel*` variants. The key kernel structure (paraphrased):

```cpp
__global__ void fused_residual_forward_kernel(
    floatX* residual, floatX* normed,            // outputs
    float* mean, float* rstd,                    // cached for backward
    const floatX* inp1, const floatX* inp2,      // residual inputs (x, attn_out)
    const floatX* weight, const floatX* bias,    // LN params
    int N, int C) {
    // one block per (b, t) row
    int idx = blockIdx.x;

    // PASS 1: each thread loads its slice and accumulates the residual sum
    //         AND computes per-thread sum and sum-of-squares for variance
    float thread_sum = 0, thread_sq = 0;
    for (int c = threadIdx.x; c < C; c += blockDim.x) {
        float r = (float)inp1[idx*C + c] + (float)inp2[idx*C + c];
        residual[idx*C + c] = (floatX)r;            // write the residual to global mem (used later)
        thread_sum += r;
        thread_sq  += r * r;
    }

    // BLOCK REDUCTIONS (warp + shared-mem trick from Chapter 13)
    float total_sum = blockReduceSum(thread_sum);
    float total_sq  = blockReduceSum(thread_sq);
    float m   = total_sum / C;
    float var = total_sq / C - m * m;
    float r   = rsqrtf(var + 1e-5f);
    if (threadIdx.x == 0) { mean[idx] = m; rstd[idx] = r; }

    // PASS 2: re-read the residual (now from cache, hopefully) and write the normalized output
    for (int c = threadIdx.x; c < C; c += blockDim.x) {
        float v = (float)residual[idx*C + c];
        normed[idx*C + c] = (floatX)(((v - m) * r) * (float)weight[c] + (float)bias[c]);
    }
}
```

What's fused:
1. Add `inp1 + inp2` (the residual)
2. Compute the row's mean and variance *in the same loop* (using the `E[X²] - (E[X])²` formula)
3. Write the normalized output

What's *not* fused: the residual `y` is still written to global memory (it's needed for backward). But it's only written once instead of being read back, so we save half a memory round-trip.


## 3. `fused_classifier` in `llm.c`

The `fused_classifier_kernel` in [`llmc/fused_classifier.cuh`](llmc/fused_classifier.cuh) is even more aggressive. Instead of running:

1. `softmax_forward` — write probs over Vp
2. `crossentropy_forward` — read probs, write losses
3. `crossentropy_softmax_backward` — read probs, write dlogits

…it does **all three** in one kernel pass over each row of logits. The pseudo-flow:

```
For each (b, t) row:
    pass 1: max-shift softmax (load logits → keep max+sum in registers)
    pass 2: compute loss = -log(prob[target])     using max, sum, logits[target]
    pass 3: compute dlogits[i] = (prob_i - 1[i==target]) * dloss   in the same loop

    Write loss (1 scalar) and dlogits (V floats). NEVER materialize probs.
```

The *probs* tensor — `(B, T, Vp)` floats, ~800 MB at GPT-2 scale — is **never written to global memory**. It's recomputed inside registers in passes 2 and 3 from `(logits[i] - max) / sum`.

Memory savings: ~800 MB. Time savings: huge (was bandwidth-bound, now compute is the same and traffic is dramatically less).

This is one of the largest single optimizations in `llm.c` and it's only possible because of the magic identity from Chapter 7: $\partial L/\partial z_i = p_i - \delta_{y, i}$. Since `dlogits` is a tiny linear function of `probs`, you can express it in terms of `(logits, max, sum, target)` directly, never needing `probs` to exist as a tensor.


## 4. Demo — Fused vs Unfused `add_then_relu`

In [ ]:
!mkdir -p course/ch15_build


In [ ]:
%%writefile course/ch15_build/fusion_demo.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// Unfused: two separate kernels, two passes through global memory
__global__ void add_kernel(float* y, const float* a, const float* b, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = a[i] + b[i];
}
__global__ void relu_kernel(float* z, const float* y, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) z[i] = fmaxf(y[i], 0.0f);
}

// Fused: a + b, immediately apply relu, write to z. y is never materialized.
__global__ void fused_add_relu(float* z, const float* a, const float* b, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float t = a[i] + b[i];
        z[i] = fmaxf(t, 0.0f);
    }
}

int main(void) {
    int N = 1 << 24;     // 16M
    float *d_a, *d_b, *d_y, *d_z;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4);
    cudaMalloc(&d_y, N*4); cudaMalloc(&d_z, N*4);
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int iters = 100;
    int block = 256, grid = (N + block - 1) / block;

    // warmup
    add_kernel<<<grid, block>>>(d_y, d_a, d_b, N);
    relu_kernel<<<grid, block>>>(d_z, d_y, N);
    fused_add_relu<<<grid, block>>>(d_z, d_a, d_b, N);
    cudaDeviceSynchronize();

    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) {
        add_kernel<<<grid, block>>>(d_y, d_a, d_b, N);
        relu_kernel<<<grid, block>>>(d_z, d_y, N);
    }
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_unf; cudaEventElapsedTime(&ms_unf, s, e);

    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) {
        fused_add_relu<<<grid, block>>>(d_z, d_a, d_b, N);
    }
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_fus; cudaEventElapsedTime(&ms_fus, s, e);

    auto bw = [N](float ms_per, int n_floats_moved){
        return (float)n_floats_moved*4.0f/1e9f / (ms_per/1000.0f);
    };
    printf("unfused (add then relu) : %.3f ms/iter  (5N moved: a, b read, y written, y read, z written)\n",
           ms_unf/iters);
    printf("                          %.1f GB/s effective (counting only useful 3N)\n",
           (3.0f*N*4.0f/1e9f) / ((ms_unf/iters)/1000.0f));
    printf("fused   (add+relu)     : %.3f ms/iter  (3N moved: a, b read, z written)\n",
           ms_fus/iters);
    printf("                          %.1f GB/s\n",
           (3.0f*N*4.0f/1e9f) / ((ms_fus/iters)/1000.0f));
    printf("speedup: %.2fx\n", ms_unf/ms_fus);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_y); cudaFree(d_z);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch15_build/fusion_demo course/ch15_build/fusion_demo.cu && ./course/ch15_build/fusion_demo


You should see the fused kernel run **somewhere between 1.5–2× faster**, depending on your GPU. The exact ratio matches the bandwidth ratio:
- unfused moves `5N` floats per iter (a, b, y read+write, then y, z read+write).
- fused moves `3N` floats per iter (a, b read once, z written once).

That's `5/3 ≈ 1.67×` more bandwidth in the unfused version. The fused version is bandwidth-bound and faster by exactly that ratio.

Now imagine doing this for every consecutive elementwise op in a Transformer block. The savings stack up.


## 5. The General Fusion Recipe

When can you fuse two ops `B = f2(f1(A))`?

| Condition | Why it matters |
|---|---|
| ✅ The intermediate result `f1(A)` isn't needed elsewhere | Otherwise you'd have to materialize it anyway |
| ✅ `f2` only reads `f1(A)` element-wise (or in the same row) | Cross-row reductions break the "stays in this block's registers" property |
| ✅ The combined kernel's register pressure stays reasonable | Too many live variables = spilling, performance loss |
| ✅ It's worth the code complexity | Trivial fusion (add+relu) vs `fused_classifier` (300 lines) — pay attention to maintenance cost |

`llm.c`'s heuristic: **always** fuse element-wise + LayerNorm (huge win, manageable code). **Sometimes** fuse the loss head (`fused_classifier`, big win, intricate code). **Don't** fuse matmul into anything unless cuBLASLt has an epilogue for it (matmul kernels are too complex to merge with custom code).


### 5a. Concrete example — the row-wise case (FUSABLE)

The second condition (*"`f2` reads `f1(A)` element-wise or same-row only"*) is the subtle one. The whole reason fusion can keep data on-chip is **how the kernel is parallelized: one block per row.** Let's make that literal.

Say `f1(A)` is a tiny `3 x 4` tensor — **3 rows** (think 3 tokens) of **C = 4** values each. The fused kernel launches **3 blocks**, and **each block owns exactly one row**:

```
                c=0   c=1   c=2   c=3        <- owned by
   block 0 -->   2     4     6     8         block 0's threads/registers
   block 1 -->   1     3     5     7         block 1's threads/registers
   block 2 -->   0     2     4     6         block 2's threads/registers
```

A block's threads load **only its own row** into registers/shared memory. Block 0 has `2, 4, 6, 8` on-chip; it has **no way to see** block 1's or block 2's values — those live in *other blocks'* private registers, and blocks can't read each other's registers or shared memory (and may not even run at the same time).

**Row-wise reduction → FUSABLE.** The LayerNorm mean of row 0 is a reduction *across that row*:

```
mean(row 0) = (2 + 4 + 6 + 8) / 4 = 5.0
```

Every value it needs (`2, 4, 6, 8`) is already in **block 0's own registers**. The reduction happens entirely on-chip (the warp + shared-mem `blockReduceSum` from Ch 13). No DRAM, no other block. ✅ This is exactly what `fused_residual_forward` does.


### 5b. The cross-row case (NOT FUSABLE on-chip)

Same `3 x 4` tensor, same one-block-per-row layout. Now suppose `f2` needs a sum *down a column* — e.g. column `c=0` across all rows:

```
                c=0   c=1   c=2   c=3
   block 0 -->   2     4     6     8
   block 1 -->   1     3     5     7
   block 2 -->   0     2     4     6

colsum(c=0) = 2 + 1 + 0 = 3
              ^   ^   ^
              |   |   block 2's register
              |   block 1's register
              block 0's register
```

The three values it needs live in **three different blocks**. No block holds more than one of them, and there is no cheap cross-block channel on-chip. The *only* place all three rows can meet is **global memory** — so you'd have to write all of `f1(A)` to DRAM and launch a second kernel to read the columns back. That write is precisely the round-trip fusion was trying to delete, so the fusion collapses. ❌

**The rule, made concrete:** fusion stays on-chip only when `f2`'s data dependency fits inside **one row** (the unit of parallelism). The moment `f2` reaches *across* rows, it reaches *across blocks*, and the only meeting place across blocks is global memory.


## 6. TODO Exercise — Fuse `add_then_gelu`

In [ ]:
%%writefile course/ch15_build/exercise1.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

__device__ float gelu(float x) {
    float cube = 0.044715f * x * x * x;
    return 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
}

// Provided unfused versions
__global__ void add_kernel(float* y, const float* a, const float* b, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = a[i] + b[i];
}
__global__ void gelu_kernel(float* z, const float* y, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) z[i] = gelu(y[i]);
}

// TODO: write a fused kernel that does z[i] = gelu(a[i] + b[i]) without materializing y.
__global__ void fused_add_gelu(float* z, const float* a, const float* b, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    // TODO: load a[i] + b[i] into a register, apply gelu, store to z[i]
}

int main(void) {
    int N = 1 << 20;
    float *h_a = (float*) malloc(N*4);
    float *h_b = (float*) malloc(N*4);
    float *h_z_unfused = (float*) malloc(N*4);
    float *h_z_fused   = (float*) malloc(N*4);
    for (int i = 0; i < N; i++) { h_a[i] = (float)((i*7)%17)/5.0f; h_b[i] = (float)((i*3)%19)/5.0f; }
    float *d_a, *d_b, *d_y, *d_z;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_y, N*4); cudaMalloc(&d_z, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);
    int block = 256, grid = (N + block - 1) / block;

    add_kernel<<<grid, block>>>(d_y, d_a, d_b, N);
    gelu_kernel<<<grid, block>>>(d_z, d_y, N);
    cudaMemcpy(h_z_unfused, d_z, N*4, cudaMemcpyDeviceToHost);

    fused_add_gelu<<<grid, block>>>(d_z, d_a, d_b, N);
    cudaMemcpy(h_z_fused, d_z, N*4, cudaMemcpyDeviceToHost);

    float maxerr = 0;
    for (int i = 0; i < N; i++) {
        float e = fabsf(h_z_unfused[i] - h_z_fused[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("max diff: %.2e %s\n", maxerr, maxerr < 1e-5 ? "PASS" : "FAIL");
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_y); cudaFree(d_z);
    free(h_a); free(h_b); free(h_z_unfused); free(h_z_fused);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch15_build/exercise1 course/ch15_build/exercise1.cu && ./course/ch15_build/exercise1


### Solution

In [ ]:
%%writefile course/ch15_build/exercise1_sol.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

__device__ float gelu(float x) {
    float cube = 0.044715f * x * x * x;
    return 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
}

__global__ void add_kernel(float* y, const float* a, const float* b, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = a[i] + b[i];
}
__global__ void gelu_kernel(float* z, const float* y, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) z[i] = gelu(y[i]);
}

__global__ void fused_add_gelu(float* z, const float* a, const float* b, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float t = a[i] + b[i];
        z[i] = gelu(t);
    }
}

int main(void) {
    int N = 1 << 20;
    float *h_a = (float*) malloc(N*4);
    float *h_b = (float*) malloc(N*4);
    float *h_z_unfused = (float*) malloc(N*4);
    float *h_z_fused   = (float*) malloc(N*4);
    for (int i = 0; i < N; i++) { h_a[i] = (float)((i*7)%17)/5.0f; h_b[i] = (float)((i*3)%19)/5.0f; }
    float *d_a, *d_b, *d_y, *d_z;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_y, N*4); cudaMalloc(&d_z, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);
    int block = 256, grid = (N + block - 1) / block;
    add_kernel<<<grid, block>>>(d_y, d_a, d_b, N);
    gelu_kernel<<<grid, block>>>(d_z, d_y, N);
    cudaMemcpy(h_z_unfused, d_z, N*4, cudaMemcpyDeviceToHost);
    fused_add_gelu<<<grid, block>>>(d_z, d_a, d_b, N);
    cudaMemcpy(h_z_fused, d_z, N*4, cudaMemcpyDeviceToHost);
    float maxerr = 0;
    for (int i = 0; i < N; i++) {
        float e = fabsf(h_z_unfused[i] - h_z_fused[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("max diff: %.2e %s\n", maxerr, maxerr < 1e-5 ? "PASS" : "FAIL");
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_y); cudaFree(d_z);
    free(h_a); free(h_b); free(h_z_unfused); free(h_z_fused);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch15_build/exercise1_sol course/ch15_build/exercise1_sol.cu && ./course/ch15_build/exercise1_sol


## Further Reading

**Source of truth**

- `llmc/fused_residual_forward.cuh` and `llmc/fused_classifier.cuh` in this repo — the production fused kernels this chapter reads.
- [cuBLAS Library documentation](https://docs.nvidia.com/cuda/cublas/) — the cuBLASLt **epilogue** (matmul + bias + activation in one launch), the canonical fusion in `llm.c`.

**Going deeper**

- [CUDA C++ Best Practices Guide — Memory Optimizations](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html) — minimizing global-memory traffic, the whole rationale for fusion.
- [_An Efficient Matrix Transpose in CUDA C/C++_](https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/) — a concrete measurement of how avoiding a global-memory round-trip changes performance.


## Recap

You now know:

- **Fusion = combining ops to skip global-memory round trips.** The combined kernel reads inputs once, computes everything in registers, writes outputs once.
- `fused_residual_forward` saves one round-trip; `fused_classifier` avoids materializing the entire `(B, T, Vp)` probs tensor.
- The fused-classifier trick depends on the simple `(probs - one_hot)` gradient identity from Chapter 7.
- Fusion is a tradeoff: ~1.5–2× speedup for elementwise+LN, but trades modularity for performance.

### What's next

**Chapter 16 — GPU Attention.** The most complex Transformer layer, on the most complex hardware. We'll see `permute_kernel`, `softmax_forward_kernel5`, and the choice between hand-rolled attention vs cuDNN Flash Attention. The attention forward is one of the hardest things to write correctly on GPU.

When you're ready, say **"proceed to Chapter 16"**.
